# Pumping from a layered coastal aquifer: two-fluid SWI and variable density (BUY)

The companion notebook [`mf6-swi-buy-a`](mf6-swi-buy-a.ipynb) compared a sharp interface (SWI) with a variable-density model (BUY) in a simple confined aquifer at steady state. This notebook takes the comparison to a more realistic coast and asks how the saltwater moves when wells start pumping:

- a cross section through a coastal plain whose layers **thicken toward the sea**, with an **upper aquifer**, a **confining unit**, and a **lower aquifer**;
- a **sloping seafloor** offshore, with recharge on the land;
- a **shallow well** in the upper aquifer and a **deep well** in the lower aquifer.

SWI runs in **two-fluid** mode, so the saltwater has to flow to where the interface moves, and its steady-state answer is the starting point of the variable-density model. Both models run a period without pumping, then 30 years of pumping, and an animation shows the saltwater in both models side by side.

> **Preview package.** SWI is not yet part of a MODFLOW 6 release. The class MODFLOW 6 includes it; see the [README](../../README.md).

## What this notebook covers

By the end of this notebook you will be able to:

- build a cross section with sloping layers, a seafloor boundary, and recharge,
- build a two-fluid SWI model with a freshwater and a saltwater model coupled by the SWI-SWI exchange,
- start a variable-density model from the SWI interface and let it settle before pumping starts,
- compare how the interface and the transition zone respond to pumping, and what each model says about the water a well pumps.

## The problem

The section runs 3 km from west to east. The coast is 2 km from the western edge; east of it the seafloor slopes down to 20 m below sea level at the eastern edge. The section is 1 m wide, so every flow is per meter of coastline, and the wells are rows of wells, or galleries, parallel to the coast.

| Unit | Top | Bottom | Horizontal $K$ (m/d) | Vertical $K$ (m/d) |
|---|---|---|---|---|
| Upper aquifer | land surface or seafloor | -20 m in the west to -40 m in the east | 10 | 1 |
| Confining unit | bottom of the upper aquifer | 5 m lower in the west, 10 m lower in the east | 0.01 | 0.01 |
| Lower aquifer | bottom of the confining unit | -60 m in the west to -120 m in the east | 20 | 2 |

- The land surface falls from 2 m in the west to sea level at the coast. Recharge of 0.0005 m/d (183 mm/yr) falls on the land.
- The sea lies on the seafloor and on the eastern edge of all three units, where the units continue under the sea. The western edge is a groundwater divide, with no flow.
- The porosity, which SWI uses as the specific yield of the interface, is 0.2 in every unit.
- The upper aquifer is treated as confined. Its top is the land surface, which is within a meter or two of the water table, so the saturated thickness changes little.
- After a period without pumping, a shallow well 250 m from the coast pumps 0.3 m³/d per meter of coastline from the upper aquifer, and a deep well 600 m from the coast pumps 0.6 m³/d per meter of coastline from the lower aquifer, for 30 years. Together they take 90 percent of the recharge, enough to draw the saltwater well inland within the 30 years.

## Imports and settings

In [ ]:
%matplotlib inline
import pathlib as pl
import time

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.collections import PolyCollection

width = 17.15 / 2.54  # figure width, inches
plt.rcParams["animation.embed_limit"] = 50  # MB, for the embedded animation

Each unit's top and bottom are a function of the distance $x$ from the western edge, evaluated at the center of each 20 m column. `tops` and `bots` hold them for the three units, which are the three layers of the SWI model.

In [ ]:
length, x_coast, delr = 3000.0, 2000.0, 20.0  # m
ncol = int(length / delr)
x = (np.arange(ncol) + 0.5) * delr
x_edges = np.arange(ncol + 1) * delr
onshore = x < x_coast

land = np.where(
    onshore, 2.0 * (1 - x / x_coast), -20.0 * (x - x_coast) / (length - x_coast)
)
bot_upper = -20.0 - 20.0 * x / length
bot_confining = bot_upper - (5.0 + 5.0 * x / length)
bot_lower = -60.0 - 60.0 * x / length
tops = np.array([land, bot_upper, bot_confining])
bots = np.array([bot_upper, bot_confining, bot_lower])
units = ["upper aquifer", "confining unit", "lower aquifer"]

kh = np.array([10.0, 0.01, 20.0])  # m/d, per unit
kv = np.array([1.0, 0.01, 2.0])
porosity, ss = 0.2, 1.0e-5
recharge = 0.0005  # m/d
delta, c_sea = 0.025, 35.0
drhodc = 1000.0 * delta / c_sea  # seawater 1,025 kg/m3

# well: (unit, distance from the western edge, rate per meter of coastline)
wells = {"shallow": (0, 1750.0, 0.3), "deep": (2, 1400.0, 0.6)}
well_col = {name: int(xw // delr) for name, (u, xw, q) in wells.items()}
years = 30.0  # of pumping
nstp = int(12 * years)  # monthly time steps

## The two-fluid SWI model

Two groundwater flow models share the three-layer grid. The **freshwater model** has the recharge, the wells, and the sea on the seafloor at the equivalent freshwater head of the sea at the seafloor, $h_f = -\delta\,z_\mathrm{seafloor}$: freshwater can discharge through the seafloor, but where saltwater fills the cell beneath the seafloor the two heads balance and no freshwater flows. The **saltwater model** has a hydraulic conductivity $1+\delta$ times that of the freshwater model, and its only boundary is the sea, at a saltwater head of 0 m on the seafloor and on the eastern edge of all three units. The SWI-SWI exchange couples the two models.

The General-Head Boundary (**GHB**) conductance on the seafloor is that of the vertical half cell below the seafloor; on the eastern edge, that of the horizontal half cell.

The first stress period is steady state, without pumping. The second is 30 years of pumping in monthly time steps. The wells have the Well Package `AUTO_FLOW_REDUCE` option, which cuts back a well when the freshwater zone of its cell becomes thinner than a tenth of the cell. The variable-density model has no such option: its wells pump their full rate throughout, whatever the water they draw.

Two-fluid SWI uses the Newton formulation, and backtracking in the Iterative Model Solution (**IMS**), as in [`synthetic-valley-h-swi`](synthetic-valley-h-swi.ipynb).

In [ ]:
swi_ws = pl.Path("models/mf6-swi-buy-b/swi-2fluid")
sim = flopy.mf6.MFSimulation(sim_ws=swi_ws, exe_name="mf6")
flopy.mf6.ModflowTdis(
    sim, nper=2, perioddata=[(1.0, 1, 1.0), (years * 365.25, nstp, 1.0)]
)
seafloor = [j for j in range(ncol) if not onshore[j]]
for name in ("fresh", "salt"):
    f = 1.0 + delta if name == "salt" else 1.0
    gwf = flopy.mf6.ModflowGwf(sim, modelname=name, newtonoptions="newton")
    flopy.mf6.ModflowGwfdis(
        gwf,
        nlay=3,
        nrow=1,
        ncol=ncol,
        delr=delr,
        delc=1.0,
        top=land,
        botm=bots.reshape(3, 1, ncol),
    )
    flopy.mf6.ModflowGwfic(gwf, strt=1.0 if name == "fresh" else 0.0)
    flopy.mf6.ModflowGwfnpf(
        gwf,
        icelltype=0,
        k=np.repeat(f * kh, ncol).reshape(3, 1, ncol),
        k33=np.repeat(f * kv, ncol).reshape(3, 1, ncol),
    )
    flopy.mf6.ModflowGwfsto(
        gwf, iconvert=0, sy=porosity, ss=ss, steady_state={0: True}, transient={1: True}
    )
    # the sea on the seafloor: the vertical half cell below it
    cond = {j: f * kv[0] * delr / (0.5 * (land[j] - bot_upper[j])) for j in seafloor}
    if name == "fresh":
        sea = [((0, 0, j), -delta * land[j], cond[j]) for j in seafloor]
        flopy.mf6.ModflowGwfghb(gwf, pname="sea", stress_period_data={0: sea})
        rch = [((0, 0, j), recharge) for j in range(ncol) if onshore[j]]
        flopy.mf6.ModflowGwfrch(gwf, stress_period_data={0: rch})
        wel = [((u, 0, well_col[w]), -q) for w, (u, xw, q) in wells.items()]
        flopy.mf6.ModflowGwfwel(
            gwf,
            stress_period_data={0: [], 1: wel},
            auto_flow_reduce=0.1,
            afrcsv_filerecord="fresh.afr.csv",
        )
    else:
        sea = [((0, 0, j), 0.0, cond[j]) for j in seafloor]
        # the eastern edge of each unit: the horizontal half cell
        sea += [
            (
                (k, 0, ncol - 1),
                0.0,
                f * kh[k] * (tops[k, -1] - bots[k, -1]) / (0.5 * delr),
            )
            for k in range(3)
        ]
        flopy.mf6.ModflowGwfghb(gwf, pname="sea", stress_period_data={0: sea})
    flopy.mf6.ModflowGwfswi(
        gwf, relative_density_difference=delta, zeta_filerecord=f"{name}.zta"
    )
    flopy.mf6.ModflowGwfoc(
        gwf, head_filerecord=f"{name}.hds", saverecord=[("HEAD", "ALL")]
    )

flopy.mf6.ModflowSwiswi(sim, exgtype="SWI6-SWI6", exgmnamea="fresh", exgmnameb="salt")
ims = flopy.mf6.ModflowIms(
    sim,
    outer_dvclose=1.0e-4,
    outer_maximum=500,
    inner_maximum=100,
    inner_dvclose=1.0e-5,
    linear_acceleration="bicgstab",
    backtracking_number=20,
    backtracking_tolerance=1.05,
    backtracking_reduction_factor=0.1,
    backtracking_residual_limit=0.002,
)
sim.register_ims_package(ims, ["fresh", "salt"])
sim.write_simulation(silent=True)
start = time.perf_counter()
success, buff = sim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print(f"SWI run time {time.perf_counter() - start:.1f} s")

Read the interface and the freshwater heads. Times count from the start of pumping, in years. `swi_toe()` gives the distance from the western edge of the farthest inland column with saltwater in a unit; the toe lies at the bottom of the unit.

In [ ]:
zeta_file = flopy.utils.HeadFile(swi_ws / "fresh.zta", text="zeta")
swi_years = (np.array(zeta_file.get_times()) - 1.0) / 365.25
swi_zeta = zeta_file.get_alldata()[:, :, 0, :]
swi_head = flopy.utils.HeadFile(swi_ws / "fresh.hds").get_alldata()[:, :, 0, :]


def swi_toe(zeta, u):
    salt = zeta[u] > bots[u] + 1.0e-3
    return x[salt].min() if salt.any() else np.nan


for u in (0, 2):
    print(
        f"{units[u]}: toe {swi_toe(swi_zeta[0], u):.0f} m from the western edge "
        f"before pumping, {swi_toe(swi_zeta[-1], u):.0f} m after {years:.0f} years"
    )
for w, (u, xw, q) in wells.items():
    j = well_col[w]
    print(
        f"{w} well: interface beneath it {swi_zeta[0, u, j]:.1f} m before pumping, "
        f"{swi_zeta[-1, u, j]:.1f} m after; {units[u]} from "
        f"{tops[u, j]:.1f} to {bots[u, j]:.1f} m"
    )
# the Well Package lists only the time steps in which a rate was reduced
afr = pd.read_csv(swi_ws / "fresh.afr.csv")
afr["years"] = (afr["time"] - 1.0) / 365.25
for number, (w, (u, xw, q)) in enumerate(wells.items(), start=1):
    cut = afr[afr["boundnumber"] == number]
    if cut.empty:
        print(f"{w} well: full rate throughout")
    else:
        print(
            f"{w} well: cut back from year {cut['years'].min():.1f}, "
            f"lowest {100 * -cut['rate-actual'].max() / q:.0f} percent of its rate"
        )

**What to look for.** Before pumping, the saltwater in the lower aquifer reaches 810 m from the western edge, 1,190 m inland of the coast: the confining unit lets only part of the recharge down into the lower aquifer, so less freshwater flows through it to hold the saltwater back, and the lower aquifer is open to the sea at its eastern edge. In the upper aquifer the saltwater reaches only 130 m inland, to 1,870 m. Pumping draws the saltwater inland and up. After 30 years the toe has moved 180 m inland in the lower aquifer and 160 m in the upper aquifer, where a layer of saltwater 3.5 m thick now lies at the bottom of the aquifer beneath the shallow well. Beneath the deep well the interface has risen 24 m, from -65.0 m to -40.8 m, only 4 m below the top of the lower aquifer, and the deep well is cut back from year 27, to 88 percent of its rate by the end. The shallow well pumps its full rate throughout. The two-fluid model runs in about a second.

## The variable-density model

The same section with 31 layers: 10 in the upper aquifer, 3 in the confining unit, and 18 in the lower aquifer, each a fixed fraction of its unit's thickness, so the layers slope with the units. The flow model gets the Buoyancy (**BUY**) Package, and a transport model carries the salt, with a longitudinal dispersivity of 1 m, a transverse dispersivity of 0.1 m, and the TVD advection scheme. The boundaries match those of the SWI model:

- the recharge carries a concentration of 0;
- the sea, a GHB with a head of 0 m and the concentration of seawater, 35 kg/m³, lies on the top layer beneath the seafloor and on the eastern edge of every layer. BUY gives the boundary the density of seawater, so the pressure of the sea grows with depth;
- the shallow well is screened through the whole upper aquifer; the deep well through the top third of the lower aquifer, above the interface. Each well's rate is split among its layers by their thickness.

**Start from SWI, then let it settle.** The transport model cannot be solved as a steady state, so the first stress period is 60 years without pumping, in 30-day time steps. It starts with seawater below the SWI interface of the steady-state period and freshwater above it, and with the SWI freshwater heads. During those 60 years mixing moves the saltwater away from the sharp interface, as in [`mf6-swi-buy-a`](mf6-swi-buy-a.ipynb). The second stress period is the same 30 years of pumping, in monthly time steps.

In [ ]:
nsub = (10, 3, 18)  # layers in each unit
unit = np.repeat([0, 1, 2], nsub)
nlay = unit.size
edges = [land]
for u, n in enumerate(nsub):
    for i in range(1, n + 1):
        edges.append(tops[u] + (bots[u] - tops[u]) * i / n)
edges = np.array(edges)  # layer tops and bottoms, nlay + 1 by ncol
z_center = 0.5 * (edges[:-1] + edges[1:])
dz = edges[:-1] - edges[1:]
screens = {"shallow": np.nonzero(unit == 0)[0], "deep": np.nonzero(unit == 2)[0][:6]}

spin_years, spin_dt = 60.0, 30.0
buy_ws = pl.Path("models/mf6-swi-buy-b/buy")
bsim = flopy.mf6.MFSimulation(sim_ws=buy_ws, exe_name="mf6")
nstp_spin = int(spin_years * 365.25 / spin_dt)
flopy.mf6.ModflowTdis(
    bsim,
    nper=2,
    perioddata=[(spin_years * 365.25, nstp_spin, 1.0), (years * 365.25, nstp, 1.0)],
)
flow = flopy.mf6.ModflowGwf(bsim, modelname="flow")
ims = flopy.mf6.ModflowIms(
    bsim,
    outer_dvclose=1.0e-5,
    outer_maximum=50,
    inner_maximum=200,
    inner_dvclose=1.0e-7,
    linear_acceleration="bicgstab",
    filename="flow.ims",
)
bsim.register_ims_package(ims, ["flow"])
dis_kw = dict(
    nlay=nlay,
    nrow=1,
    ncol=ncol,
    delr=delr,
    delc=1.0,
    top=land,
    botm=edges[1:].reshape(nlay, 1, ncol),
)
flopy.mf6.ModflowGwfdis(flow, **dis_kw)
flopy.mf6.ModflowGwfic(flow, strt=swi_head[0][unit].reshape(nlay, 1, ncol))
flopy.mf6.ModflowGwfnpf(
    flow,
    icelltype=0,
    k=np.repeat(kh[unit], ncol).reshape(nlay, 1, ncol),
    k33=np.repeat(kv[unit], ncol).reshape(nlay, 1, ncol),
)
flopy.mf6.ModflowGwfsto(flow, iconvert=0, ss=ss, transient={0: True})
flopy.mf6.ModflowGwfbuy(
    flow, packagedata=[(0, drhodc, 0.0, "transport", "concentration")]
)
rch = [((0, 0, j), recharge, 0.0) for j in range(ncol) if onshore[j]]
flopy.mf6.ModflowGwfrch(
    flow, auxiliary="concentration", pname="rch", stress_period_data={0: rch}
)
sea = [((0, 0, j), 0.0, kv[0] * delr / (0.5 * dz[0, j]), c_sea) for j in seafloor]
sea += [
    ((k, 0, ncol - 1), 0.0, kh[unit[k]] * dz[k, -1] / (0.5 * delr), c_sea)
    for k in range(nlay)
]
flopy.mf6.ModflowGwfghb(
    flow, auxiliary="concentration", pname="sea", stress_period_data={0: sea}
)
wel = []
for w, (u, xw, q) in wells.items():
    j, ks = well_col[w], screens[w]
    share = dz[ks, j] / dz[ks, j].sum()
    wel += [((k, 0, j), -q * s, 0.0) for k, s in zip(ks, share)]
flopy.mf6.ModflowGwfwel(
    flow, auxiliary="concentration", pname="wel", stress_period_data={0: [], 1: wel}
)

transport = flopy.mf6.ModflowGwt(bsim, modelname="transport")
imt = flopy.mf6.ModflowIms(
    bsim,
    outer_dvclose=1.0e-4,
    outer_maximum=100,
    inner_maximum=300,
    inner_dvclose=1.0e-6,
    linear_acceleration="bicgstab",
    filename="transport.ims",
)
bsim.register_ims_package(imt, ["transport"])
flopy.mf6.ModflowGwtdis(transport, **dis_kw)
# seawater below the SWI interface of the steady-state period, freshwater above
strt = np.where(z_center < swi_zeta[0][unit], c_sea, 0.0)
flopy.mf6.ModflowGwtic(transport, strt=strt.reshape(nlay, 1, ncol))
flopy.mf6.ModflowGwtmst(transport, porosity=porosity)
flopy.mf6.ModflowGwtadv(transport, scheme="tvd")
flopy.mf6.ModflowGwtdsp(transport, alh=1.0, ath1=0.1, diffc=8.6e-5)
flopy.mf6.ModflowGwtssm(
    transport,
    sources=[
        ("sea", "aux", "concentration"),
        ("wel", "aux", "concentration"),
        ("rch", "aux", "concentration"),
    ],
)
flopy.mf6.ModflowGwtoc(
    transport,
    concentration_filerecord="transport.ucn",
    saverecord={
        0: [("CONCENTRATION", "FREQUENCY", 122), ("CONCENTRATION", "LAST")],
        1: [("CONCENTRATION", "FREQUENCY", 6)],
    },
)
flopy.mf6.ModflowGwfgwt(bsim, exgmnamea="flow", exgmnameb="transport")
bsim.write_simulation(silent=True)
start = time.perf_counter()
success, buff = bsim.run_simulation(silent=True)
assert success, "MODFLOW 6 did not terminate normally"
print(f"BUY run time {time.perf_counter() - start:.0f} s")

Read the relative concentration, 0 for freshwater and 1 for seawater, saved every 10 years of the first period and every 6 months of pumping. `buy_toe()` is the farthest inland column where the bottom layer of a unit is at least half seawater, and `pumped_fraction()` the seawater fraction of the water a well pumps, the average over its screen weighted by the rate of each layer.

In [ ]:
ucn = flopy.utils.HeadFile(buy_ws / "transport.ucn", text="concentration")
buy_years = np.array(ucn.get_times()) / 365.25 - spin_years
conc = ucn.get_alldata()[:, :, 0, :] / c_sea


def buy_toe(c, u):
    bottom = c[np.nonzero(unit == u)[0][-1]]
    return x[bottom >= 0.5].min() if (bottom >= 0.5).any() else np.nan


def pumped_fraction(c, w):
    j, ks = well_col[w], screens[w]
    return (c[ks, j] * dz[ks, j]).sum() / dz[ks, j].sum()


print("years   50 percent toe (m)       seawater fraction pumped")
print("        upper    lower           shallow   deep")
for t, c in zip(buy_years, conc):
    if t <= 0.0 or abs(t % 5.0) < 0.01:
        print(
            f"{t:6.1f}  {buy_toe(c, 0):6.0f}   {buy_toe(c, 2):6.0f}"
            f"          {pumped_fraction(c, 'shallow'):6.3f}    "
            f"{pumped_fraction(c, 'deep'):6.3f}"
        )

**What to look for.** In the 60 years without pumping, mixing moves the saltwater toward the sea, as in [`mf6-swi-buy-a`](mf6-swi-buy-a.ipynb): the 50 percent line at the bottom of the lower aquifer moves from 810 m, where SWI put the toe, to 990 m, quickly at first and only about 2 m a year by the end. In the upper aquifer it moves to the coast. Once pumping starts, the 50 percent toes move inland by about 100 m in each aquifer, and the water the deep well pumps gets saltier year after year: 1.0 percent seawater before pumping, 6.6 percent after 5 years, 10 percent after 10, and 20 percent after 30, about 7 kg/m³ of salt. The shallow well stays below 1 percent. The variable-density run takes about 40 seconds.

## Draw the two models

`draw()` draws one panel: the SWI interface in each unit, or the concentration of the variable-density model, with the sea above the seafloor, the unit boundaries, and the well screens. The SWI panel fills each cell orange below the interface and blue above it; the variable-density panel colors each cell by its fraction of seawater on a scale from blue to orange.

In [ ]:
fresh_color, salt_color, sea_color = "#9ecae1", "#fdae6b", "0.85"
cmap = mpl.colors.LinearSegmentedColormap.from_list("fs", [fresh_color, salt_color])
x_grid = np.broadcast_to(x, z_center.shape)  # cell centers, for contouring


def boxes(top, bot):
    # one rectangle for each cell, from its bottom to its top
    x0, x1 = x_edges[:-1], x_edges[1:]
    return [
        [(a, b), (c, b), (c, t), (a, t)]
        for row_t, row_b in zip(np.atleast_2d(top), np.atleast_2d(bot))
        for a, c, t, b in zip(x0, x1, row_t, row_b)
    ]


def draw(ax, kind):
    sea_top = np.where(onshore, np.nan, 0.0)
    ax.fill_between(x, land, sea_top, step="mid", color=sea_color, lw=0)
    if kind == "swi":
        artists = [
            PolyCollection([], color=salt_color, lw=0),
            PolyCollection([], color=fresh_color, lw=0),
        ]
    else:
        artists = [PolyCollection(boxes(edges[:-1], edges[1:]), cmap=cmap, lw=0)]
        artists[0].set_clim(0.0, 1.0)
    for a in artists:
        ax.add_collection(a)
    # where the saltwater was before pumping, dashed
    if kind == "swi":
        z0 = swi_zeta[0]
        for k in range(3):
            inside = (z0[k] > bots[k] + 1.0e-3) & (z0[k] < tops[k] - 1.0e-3)
            ax.step(
                x,
                np.where(inside, z0[k], np.nan),
                where="mid",
                color="black",
                ls="--",
                lw=1.0,
            )
    else:
        ax.contour(
            x_grid,
            z_center,
            conc[buy_years <= 0.0][-1],
            levels=[0.5],
            colors="black",
            linestyles="--",
            linewidths=1.0,
        )
        artists.append(None)  # the 50 percent line now, drawn by update()
    for surface in (land, bot_upper, bot_confining, bot_lower):
        ax.step(x, surface, where="mid", color="black", lw=0.5)
    for w, (u, xw, q) in wells.items():
        j, ks = well_col[w], screens[w]
        ax.plot(
            [x[j], x[j]],
            [edges[ks[0], j], edges[ks[-1] + 1, j]],
            color="purple",
            lw=2.5,
        )
        ax.text(x[j], land[j] + 1.5, f"{w}\nwell", ha="center", va="bottom", fontsize=7)
    ax.text(2600, -5, "sea", ha="center", fontsize=8)
    ax.text(
        150,
        0.5 * (bot_upper[7] + bot_confining[7]),
        "confining unit",
        fontsize=7,
        va="center",
    )
    ax.set_xlim(0.0, length)
    ax.set_ylim(-120.0, 12.0)
    ax.set_ylabel("Elevation (m)")
    return artists


def update(artists, kind, values):
    if kind == "swi":
        z = np.clip(values, bots, tops)
        artists[0].set_verts(boxes(z, bots))
        artists[1].set_verts(boxes(tops, z))
    else:
        artists[0].set_array(values.ravel())
        if artists[1] is not None:
            artists[1].remove()
        artists[1] = artists[0].axes.contour(
            x_grid, z_center, values, levels=[0.5], colors="black", linewidths=1.0
        )


def two_panels(height=0.85):
    fig, axd = plt.subplot_mosaic(
        [["swi"], ["buy"]],
        figsize=(width, height * width),
        layout="constrained",
        sharex=True,
    )
    panels = {}
    for kind, title in (
        ("swi", "Two-fluid SWI"),
        ("buy", "Variable density (BUY)"),
    ):
        panels[kind] = draw(axd[kind], kind)
        axd[kind].set_title(title, loc="left", fontsize=9, fontweight="bold")
    axd["buy"].set_xlabel("Distance from the western edge (m)")
    fig.colorbar(
        panels["buy"][0],
        ax=[axd["swi"], axd["buy"]],
        shrink=0.6,
        label="Fraction of seawater (variable density)",
    )
    handles = [
        mpl.patches.Patch(color=fresh_color, label="Freshwater"),
        mpl.patches.Patch(color=salt_color, label="Saltwater"),
        mpl.patches.Patch(color=sea_color, label="Sea"),
        mpl.lines.Line2D([], [], color="purple", lw=2.5, label="Well screen"),
        mpl.lines.Line2D([], [], color="black", lw=1.0, label="50 percent seawater"),
        mpl.lines.Line2D(
            [],
            [],
            color="black",
            ls="--",
            lw=1.0,
            label="Interface or 50 percent line before pumping",
        ),
    ]
    fig.legend(handles=handles, loc="outside lower center", ncol=3, frameon=False)
    return fig, axd, panels

### Before pumping

The SWI interface at the end of the steady-state period, and the variable-density model at the end of its 60 years without pumping.

In [ ]:
fig, axd, panels = two_panels()
update(panels["swi"], "swi", swi_zeta[0])
update(panels["buy"], "buy", conc[buy_years <= 0.0][-1])
fig.suptitle("Before pumping", fontsize=10)
plt.show()

**What to look for.** The two models agree on the overall picture: a long wedge of saltwater in the lower aquifer beneath the confining unit, and a short one in the upper aquifer at the coast. They differ in the details. In the lower aquifer the variable-density model has a transition zone tens of meters thick, and its 50 percent line reaches 180 m less far inland than the SWI interface. In the upper aquifer the freshwater flowing out to the sea keeps the variable-density model fresh to the coast and in the first 30 m offshore, and it flushes brackish water up through the seafloor, while the SWI interface reaches 130 m inland. Both models have saltwater in the confining unit beneath the sea and the coast.

### The interface beneath the deep well, and what the wells pump

Plot through time the elevation of the SWI interface beneath the deep well and of the 50 percent seawater line of the variable-density model, found by interpolating between layer centers, and the seawater fraction of the water each well pumps in the variable-density model.

In [ ]:
def line_elevation(c, j, u, level=0.5):
    # elevation where the concentration in unit u of column j first reaches level,
    # going down from the top of the unit
    ks = np.nonzero(unit == u)[0]
    col, zc = c[ks, j], z_center[ks, j]
    if col[-1] < level:
        return bots[u, j]
    if col[0] >= level:
        return tops[u, j]
    i = np.argmax(col >= level)
    return np.interp(level, [col[i - 1], col[i]], [zc[i - 1], zc[i]])


pumping = buy_years >= 0.0
j = well_col["deep"]
with flopy.plot.styles.USGSPlot():
    fig, axd = plt.subplot_mosaic(
        [["A", "B", "C"]], figsize=(width, 0.42 * width), layout="constrained"
    )
    ax = axd["A"]
    ax.plot(swi_years, swi_zeta[:, 2, j], color="black", label="SWI interface")
    ax.plot(
        buy_years[pumping],
        [line_elevation(c, j, 2) for c in conc[pumping]],
        color="black",
        ls="--",
        label="50 percent seawater",
    )
    ax.axhspan(
        edges[screens["deep"][-1] + 1, j], tops[2, j], color="purple", alpha=0.15
    )
    ax.set_xlim(0, years)
    ax.set_ylim(bots[2, j], tops[2, j])
    ax.set_xlabel("Years since pumping started")
    ax.set_ylabel("Elevation (m)")
    flopy.plot.styles.graph_legend(ax=ax, loc="lower right")
    flopy.plot.styles.heading(ax=ax, letter="A", heading="Below the deep well")
    ax = axd["B"]
    for number, (w, (u, xw, q)) in enumerate(wells.items(), start=1):
        rate = np.full(swi_years.size, 100.0)
        for t, r in afr[afr["boundnumber"] == number][["years", "rate-actual"]].values:
            rate[np.argmin(abs(swi_years - t))] = 100.0 * -r / q
        ax.plot(
            swi_years[1:],
            rate[1:],
            color="purple",
            ls=":" if w == "shallow" else "-",
            label=f"{w} well",
        )
    ax.set_xlim(0, years)
    ax.set_ylim(0, 105)
    ax.set_xlabel("Years since pumping started")
    ax.set_ylabel("Rate pumped (% of requested)")
    flopy.plot.styles.graph_legend(ax=ax, loc="lower left")
    flopy.plot.styles.heading(ax=ax, letter="B", heading="SWI: pumping rate")
    ax = axd["C"]
    for w, ls in (("shallow", ":"), ("deep", "-")):
        ax.plot(
            buy_years[pumping],
            [100 * pumped_fraction(c, w) for c in conc[pumping]],
            color="purple",
            ls=ls,
            label=f"{w} well",
        )
    ax.set_xlim(0, years)
    ax.set_ylim(0, None)
    ax.set_xlabel("Years since pumping started")
    ax.set_ylabel("Seawater in the pumped water (%)")
    flopy.plot.styles.graph_legend(ax=ax, loc="upper left")
    flopy.plot.styles.heading(ax=ax, letter="C", heading="BUY: pumped water")
    plt.show()

**What to look for.** Beneath the deep well (A) the SWI interface rises 24 m in 30 years and the 50 percent line 19 m, both fastest in the first years. The SWI interface reaches the bottom of the screen, -53.9 m (the purple band), after 9 years, and the 50 percent line only after about 20 years. But the transition zone above the 50 percent line gets there first: the 10 percent line crosses the bottom of the screen within 2 years. So the two models tell the story of the deep well differently. SWI pumps the full rate of freshwater for 27 years and then cuts the well back (B), when the freshwater above the interface runs thin. The variable-density well pumps its full rate throughout, but the water gets saltier from the first year (C), 20 percent seawater after 30 years. The shallow well, with only a thin layer of saltwater reaching beneath it late in the run, pumps its full rate of almost fresh water in both models.

## Animation: 30 years of pumping

Every frame is 6 months of pumping. Use the play button or the slider to step through the run. The animation is also saved as a GIF, `models/mf6-swi-buy-b/animation.gif`, for slides.

In [ ]:
frames = np.nonzero(pumping)[0]
fig, axd, panels = two_panels()
stamp = fig.suptitle("", fontsize=10)


def frame(i):
    t = buy_years[i]
    update(panels["swi"], "swi", swi_zeta[np.argmin(abs(swi_years - t))])
    update(panels["buy"], "buy", conc[i])
    stamp.set_text(f"{t:4.1f} years after pumping started")
    return []


anim = FuncAnimation(fig, frame, frames=frames, interval=200)
anim.save("models/mf6-swi-buy-b/animation.gif", writer=PillowWriter(fps=5), dpi=100)
plt.close(fig)
HTML(anim.to_jshtml())

**What to look for.** The dashed lines show where the saltwater was before pumping. In the SWI panel the interface rises in a broad cone beneath the deep well, the wedge in the lower aquifer moves inland, and the saltwater in the upper aquifer creeps under the shallow well. In the variable-density panel the 50 percent line (solid) follows the same path a little behind the SWI interface, and the brackish water above it reaches the deep well screen within the first years.

### Try it

- **Pump less.** Halve both rates in `wells` and run the notebook again. Does SWI still cut back the deep well? How much seawater does the variable-density well pump after 30 years?
- **Move the deep well.** Move it inland, to 1,000 m from the western edge, closer to the toe of the wedge. Which model is more sensitive to the move?
- **Less mixing.** Set the dispersivities to 0.1 and 0.01 m. Does the variable-density interface move toward the SWI interface before pumping, and does the deep well pump less seawater?

## Recap

You built a coastal cross section with sloping units, a confining unit between two aquifers, a sloping seafloor, and recharge, and solved it two ways. Two-fluid SWI used one layer per unit and a freshwater and a saltwater model coupled by the SWI-SWI exchange, and ran in about a second. The variable-density model used 31 layers, started from the SWI interface, and needed a 60-year period without pumping to let mixing settle the transition zone before the 30 years of pumping.

Both models move the saltwater toward the wells along the same path and at a similar pace, but they answer different questions. SWI tells where the interface is, whether it reaches a well, how fast it moves, and when a well has to be cut back, in a model with one layer per aquifer. The variable-density model also tells how salty the water a well pumps becomes long before the interface arrives, because the transition zone, not the 50 percent line, reaches the well screen first.